# Library tour

This lab tries out a little of each library in the chapter: NumPy, SciPy, Matplotlib, pandas, scikit-learn and ElementTree. The exercises are independent, except when they reuse an earlier exercise's data, and go from the simplest to the longest: the last one is optional. All the data is generated in the notebook, there is nothing to download.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

## 1. NumPy: indexing an array

The `durations` array gives request durations, in milliseconds: one row per server (4 servers), one column per hour (6 hours).

1. Print its shape.
2. Print the duration of the third server at the last hour, then the first server's row, then the second hour's column.
3. With a boolean mask, print the durations above 200 ms, and count them.
4. Compute each server's mean duration, then find the slowest one with `argmax`.

**Useful documentation**

- [`numpy.ndarray.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.argmax.html)

In [ ]:
durations = np.array(
  [
    [120, 95, 310, 88, 140, 450],
    [110, 105, 130, 98, 125, 180],
    [240, 260, 230, 290, 310, 275],
    [90, 85, 100, 92, 88, 97],
  ]
)

# Your code here

### Additional hints

Indices start at 0: the third server is row 2, and `-1` is the last element. `durations[:, 1]` takes all the rows of column 1. `axis=1` computes along the columns, so one mean per row. You should find 8 durations above 200 ms, and the server at index 2 as the slowest, with a mean of 267.5 ms.

```python
print(durations.shape)
print(durations[2, -1])
print(durations[0])
print(durations[:, 1])
slow = durations[...]
print(slow, len(slow))
means = durations.mean(axis=...)
print(means, means.argmax())
```

### Solution

In [ ]:
print("Shape:", durations.shape)
print("Third server, last hour:", durations[2, -1])
print("First server:", durations[0])
print("Second hour:", durations[:, 1])
slow = durations[durations > 200]
print("Durations above 200 ms:", slow)
print("Count:", len(slow))
means = durations.mean(axis=1)
print("Means:", means)
print("Slowest server:", means.argmax())

## 2. SciPy: testing and optimizing

The `before` and `after` lists give request durations, in milliseconds, before and after a deployment.

1. Summarize `before` with `stats.describe`, and print the two means.
2. Compare the two series with `stats.ttest_ind`, without assuming equal variances. At the 5% level, is the drop real?
3. The `cost` function gives the running cost for `n` servers. With `optimize.minimize_scalar`, find the number of servers that minimizes it, between 1 and 50.
4. A number of servers is an integer: compare `cost` at the two nearest integers.

**Useful documentation**

- [`scipy.stats.describe`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.describe.html)

In [ ]:
before = [132, 118, 125, 140, 151, 122, 137, 129, 145, 120]
after = [118, 110, 127, 105, 121, 115, 108, 124, 112, 119]


def cost(n: float) -> float:
  return 80 * n + 5000 / n


# Your code here

### Additional hints

`equal_var=False` asks for Welch's test, which doesn't assume equal variances. The result has a `pvalue` field. `bounds=(1, 50)` bounds the search, and the result gives the position of the minimum in `.x`, its value in `.fun`. You should find a p-value of about 0.0016, and an optimum at 7.91 servers for a cost of 1264.9.

```python
print(stats.describe(before))
print(np.mean(before), np.mean(after))
test = stats.ttest_ind(before, after, equal_var=...)
print(test.pvalue)
optimum = optimize.minimize_scalar(cost, bounds=...)
print(optimum.x, optimum.fun)
print(cost(7), cost(8))
```

### Solution

In [ ]:
print(stats.describe(before))
print("Mean before:", np.mean(before))
print("Mean after:", np.mean(after))
test = stats.ttest_ind(before, after, equal_var=False)
print(f"p-value: {test.pvalue:.4f}")
optimum = optimize.minimize_scalar(cost, bounds=(1, 50))
print(f"Optimum: {optimum.x:.2f} servers, cost {optimum.fun:.1f}")
print(f"7 servers: {cost(7):.1f}, 8 servers: {cost(8):.1f}")

The p-value is below 0.05: such a gap would be unlikely if the two means were equal, so the drop is real at this level. Among the integers, 8 servers cost the least.

## 3. Matplotlib: two charts

With the `durations` array from exercise 1:

1. Create a figure with two charts side by side, with `plt.subplots`.
2. On the left, draw one curve per server, with a legend, and a horizontal line at the 200 ms threshold. Name the axes.
3. On the right, draw the histogram of all the durations, with 10 bins. Name the axes.
4. Give each chart a title, and save the figure to `durations.png`.

**Useful documentation**

- [`Axes.hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.hist.html)
- [`Axes.set_title`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_title.html)
- [`numpy.ndarray.ravel`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.ravel.html)

In [ ]:
# Your code here

### Additional hints

`plt.subplots(1, 2, figsize=(10, 4))` returns the figure and an array of two `Axes`. `enumerate(durations)` goes through the rows with their index, for the legend. `durations.ravel()` flattens the array for the histogram, and `bins=10` sets the number of bins.

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, row in enumerate(durations):
  axes[0].plot(row, marker="o", label=f"server {i}")
axes[0].axhline(...)
axes[0].legend()
axes[1].hist(durations.ravel(), bins=...)
fig.savefig("durations.png")
```

### Solution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, row in enumerate(durations):
  axes[0].plot(row, marker="o", label=f"server {i}")
axes[0].axhline(200, color="red", linestyle="--", label="threshold")
axes[0].set_xlabel("hour")
axes[0].set_ylabel("duration (ms)")
axes[0].set_title("Durations (ms) per server")
axes[0].legend()
axes[1].hist(durations.ravel(), bins=10)
axes[1].set_xlabel("duration (ms)")
axes[1].set_ylabel("number of requests")
axes[1].set_title("Distribution of durations")
fig.savefig("durations.png")

## 4. pandas: filtering and grouping

The `requests` `DataFrame` describes 300 simulated requests: their service, the server's load, their duration and their HTTP status.

1. Look at the first rows, and summarize the numeric columns with `describe`.
2. Count the requests of each status with `value_counts`.
3. Keep the failed requests (status 500) over 300 ms, and count them.
4. Add a `seconds` column, the duration in seconds.
5. Compute each service's mean duration.

**Useful documentation**

- [`DataFrame.head`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)
- [`DataFrame.describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

In [ ]:
rng = np.random.default_rng(0)
load = rng.integers(1, 60, size=300)
requests = pd.DataFrame(
  {
    "service": rng.choice(["api", "web", "batch"], size=300),
    "load": load,
    "duration_ms": (60 + 6.5 * load + rng.normal(0, 20, size=300)).round(),
    "status": rng.choice([200, 404, 500], size=300, p=[0.85, 0.1, 0.05]),
  }
)

# Your code here

### Additional hints

In a cell, only the last expression shows: use `print` or `display` for the others. A condition on two columns is written with `&`, each comparison in parentheses. `requests.groupby("service")["duration_ms"].mean()` gives one mean per service. You should find 5 failed requests over 300 ms.

```python
print(requests.head())
print(requests.describe())
print(requests["status"].value_counts())
slow = requests[(...) & (...)]
print(len(slow))
requests["seconds"] = ...
print(requests.groupby("service")["duration_ms"].mean())
```

### Solution

In [ ]:
print(requests.head())
print(requests.describe())
print(requests["status"].value_counts())
slow = requests[(requests["status"] == 500) & (requests["duration_ms"] > 300)]
print("Slow failed requests:", len(slow))
requests["seconds"] = requests["duration_ms"] / 1000
print(requests.groupby("service")["duration_ms"].mean())

## 5. scikit-learn: predicting a duration

With the `requests` `DataFrame` from the previous exercise, predict a request's duration from the load. `X` and `y` are given.

1. Split the data into training and test sets with `train_test_split`, with `random_state=0`.
2. Train a `LinearRegression`, and print its score on the test set.
3. Read its slope and intercept, to compare with the simulation's formula, then predict the duration for a load of 30.
4. Replace the model with a `DecisionTreeRegressor`, with `random_state=0`, and compare its scores on the training and test sets.

**Useful documentation**

- [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [`DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)

In [ ]:
X = requests[["load"]].to_numpy()
y = requests["duration_ms"].to_numpy()

# Your code here

### Additional hints

`train_test_split(X, y, random_state=0)` returns four arrays, in the order `X_train, X_test, y_train, y_test`. The trained model has the attributes `coef_` (an array with one slope per column of `X`) and `intercept_`. `predict` expects a 2D array: `[[30]]`, one row, one column. The simulation uses `60 + 6.5 * load`, so the prediction for a load of 30 should be close to 255 ms.

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
model = LinearRegression().fit(...)
print(model.score(X_test, y_test))
print(model.coef_[0], model.intercept_)
print(model.predict([[30]])[0])
tree = DecisionTreeRegressor(random_state=0).fit(...)
print(tree.score(X_train, y_train), tree.score(X_test, y_test))
```

### Solution

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
model = LinearRegression().fit(X_train, y_train)
print(f"Test R²: {model.score(X_test, y_test):.3f}")
print(f"Slope: {model.coef_[0]:.2f}, intercept: {model.intercept_:.1f}")
print(f"Load 30: {model.predict([[30]])[0]:.0f} ms")
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print(
  f"Tree, training: {tree.score(X_train, y_train):.3f}, "
  f"test: {tree.score(X_test, y_test):.3f}"
)

Changing the algorithm only took changing the class. The tree scores better on the training set than on the test set, where it does worse than the line: only the test score estimates what a model will be worth on new requests.

## 6. ElementTree: reading XML

The `log` string holds a request log in XML format.

1. Parse it with `ET.fromstring`, and print the root's name and its number of children.
2. Go through the requests, and print each one's service and duration.
3. With `findall`, keep the failed requests (status 500), and compute their mean duration.
4. Count the requests of each service with `Counter`.

**Useful documentation**

- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)

In [ ]:
log = """<log>
  <request service="api" status="200"><time>120</time></request>
  <request service="web" status="200"><time>95</time></request>
  <request service="web" status="500"><time>310</time></request>
  <request service="api" status="200"><time>88</time></request>
  <request service="batch" status="200"><time>640</time></request>
  <request service="api" status="500"><time>450</time></request>
</log>"""

# Your code here

### Additional hints

`root.tag` gives the name, `len(root)` the number of children. An attribute is read with `.get("service")`, a child's text with `.findtext("time")`, always as a string: convert it with `int`. The path `"request[@status='500']"` filters on an attribute. You should find a mean duration of 380 ms for the errors, and 3 requests for `api`.

```python
root = ET.fromstring(log)
print(root.tag, len(root))
for request in root:
  print(request.get(...), request.findtext(...))
errors = root.findall(...)
print(np.mean([int(e.findtext("time")) for e in errors]))
print(Counter(request.get("service") for request in root))
```

### Solution

In [ ]:
root = ET.fromstring(log)
print("Root:", root.tag)
print("Requests:", len(root))
for request in root:
  print(request.get("service"), int(request.findtext("time")))
errors = root.findall("request[@status='500']")
print("Mean duration of errors:", np.mean([int(e.findtext("time")) for e in errors]))
print(Counter(request.get("service") for request in root))

## 7. To finish: from the XML log to the chart

This optional exercise chains three libraries. The cell below writes `log.xml`, one day's log: 500 requests, with their service, hour, status and duration.

1. Read the file with `ET.parse`, and build a `DataFrame` with one row per request, with numeric columns for the hour, the status and the duration.
2. Keep the successful requests (status 200), and compute their mean duration for each hour.
3. Plot this mean against the hour, with Matplotlib. At what time is the service slowest?

**Useful documentation**

- [`ET.parse`](https://docs.python.org/3/library/xml.etree.elementtree.html#xml.etree.ElementTree.parse)
- [`ElementTree.getroot`](https://docs.python.org/3/library/xml.etree.elementtree.html#xml.etree.ElementTree.ElementTree.getroot)

In [ ]:
rng = np.random.default_rng(1)
root = ET.Element("log")
for _ in range(500):
  hour = int(rng.integers(0, 24))
  load = 10 + 40 * np.exp(-((hour - 14) ** 2) / 18)
  request = ET.SubElement(
    root,
    "request",
    service=str(rng.choice(["api", "web"])),
    hour=str(hour),
    status=str(rng.choice([200, 500], p=[0.9, 0.1])),
  )
  ET.SubElement(request, "time").text = str(round(60 + 6.5 * load + rng.normal(0, 20)))
ET.ElementTree(root).write("log.xml", encoding="utf-8", xml_declaration=True)

# Your code here

### Additional hints

`ET.parse("log.xml").getroot()` gives the root. A list of dictionaries, one per request, goes straight into `pd.DataFrame`. Convert the numbers with `int` while reading the XML. Then `groupby("hour")["duration_ms"].mean()` gives a `Series` whose index is the hour, to pass to `ax.plot(per_hour.index, per_hour)`. The maximum is around 14:00.

```python
root = ET.parse("log.xml").getroot()
rows = [
  {
    "hour": int(request.get("hour")),
    "status": ...,
    "duration_ms": ...,
  }
  for request in root
]
log_df = pd.DataFrame(rows)
ok = log_df[...]
per_hour = ok.groupby("hour")["duration_ms"].mean()
fig, ax = plt.subplots()
ax.plot(per_hour.index, per_hour, marker="o")
```

### Solution

In [ ]:
root = ET.parse("log.xml").getroot()
rows = [
  {
    "service": request.get("service"),
    "hour": int(request.get("hour")),
    "status": int(request.get("status")),
    "duration_ms": int(request.findtext("time")),
  }
  for request in root
]
log_df = pd.DataFrame(rows)
ok = log_df[log_df["status"] == 200]
per_hour = ok.groupby("hour")["duration_ms"].mean()
fig, ax = plt.subplots()
ax.plot(per_hour.index, per_hour, marker="o")
ax.set_xlabel("hour")
ax.set_ylabel("duration (ms)")
print(per_hour.idxmax())